# Generate an image batch from a prompt

Choose SDXL Base or SDXL Turbo, enter a prompt, and click **Submit SLURM job**. The notebook submits a GPU job from the cluster login node; it does not generate on the notebook kernel. Click **Refresh status / show images** to check progress and display saved images. Each run writes its settings under `run_outputs/`.

Open this notebook in Jupyter on the cluster login node, where `sbatch` is available. Generation runs on a SLURM GPU node. Model weights must be available in the shared Hugging Face cache (or downloadable from the compute node).

In [ ]:
from pathlib import Path
import sys
import json
import subprocess
from datetime import datetime
from IPython.display import display, clear_output, Image as DisplayImage
import ipywidgets as widgets

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "generation/generation_utils.py").exists())
SLURM_SCRIPT = ROOT / "generation/generate_batch.slurm"
RUN_ROOT = ROOT / "run_outputs"
CONFIG_DIR = RUN_ROOT / "configs"
IMAGE_ROOT = RUN_ROOT / "images"
LOG_DIR = RUN_ROOT / "logs"
for folder in (CONFIG_DIR, IMAGE_ROOT, LOG_DIR):
    folder.mkdir(parents=True, exist_ok=True)

if subprocess.run(["bash", "-lc", "command -v sbatch"], capture_output=True).returncode:
    raise RuntimeError("sbatch is not available. Open this notebook in Jupyter on the cluster login node.")

jobs = {}

def submit_job(model_name, prompt_text, image_count, first_seed, inference_steps, guidance_scale, run_name):
    config = {
        "model_name": model_name,
        "prompt": prompt_text,
        "count": int(image_count),
        "seed": int(first_seed),
        "steps": int(inference_steps),
        "guidance": float(guidance_scale),
        "output_root": str(IMAGE_ROOT),
        "run_name": run_name,
    }
    config_path = CONFIG_DIR / f"{run_name}.json"
    config_path.write_text(json.dumps(config, indent=2) + "\n")
    result = subprocess.run(
        ["sbatch", "--parsable", str(SLURM_SCRIPT), str(config_path)],
        cwd=ROOT, text=True, capture_output=True, check=True,
    )
    job_id = result.stdout.strip().split(";")[0]
    jobs[job_id] = IMAGE_ROOT / run_name
    print(f"Submitted {model_name} job {job_id}; output: {jobs[job_id]}")
    print(f"SLURM log: {LOG_DIR / f'sdxl_batch-{job_id}.out'}")
    return job_id

def refresh_jobs(_):
    with output:
        clear_output(wait=True)
        if not jobs:
            print("Submit a generation job first.")
            return
        for job_id, output_dir in jobs.items():
            queue = subprocess.run(["squeue", "-h", "-j", job_id, "-o", "%T"],
                                   text=True, capture_output=True)
            state = queue.stdout.strip()
            if not state:
                accounting = subprocess.run(["sacct", "-n", "-X", "-j", job_id, "--format=State"],
                                            text=True, capture_output=True)
                state = next((line.strip() for line in accounting.stdout.splitlines() if line.strip()), "No status returned")
            print(f"Job {job_id}: {state}")
            images = sorted(output_dir.glob("image_*.png"))
            if images:
                display(*[DisplayImage(filename=str(p)) for p in images])
            elif state.startswith("FAILED") or state.startswith("CANCELLED") or state.startswith("OUT_OF_MEMORY"):
                log = LOG_DIR / f"sdxl_batch-{job_id}.out"
                if log.exists():
                    print("Last log lines:\n" + "\n".join(log.read_text(errors="replace").splitlines()[-30:]))


## Set up the batch

Turbo uses one step and zero guidance. Base uses 30 steps, guidance 5, and 1024 × 1024 output. Images are generated sequentially to limit GPU memory use. The job log, JSON settings, and output images are saved under `run_outputs/`.

In [ ]:
model = widgets.Dropdown(options=["SDXL Base", "SDXL Turbo"], value="SDXL Base", description="Model:")
prompt = widgets.Textarea(value="A photo of a flower in a garden", description="Prompt:", layout=widgets.Layout(width="90%", height="90px"))
count = widgets.IntSlider(value=4, min=1, max=12, step=1, description="Images:")
seed = widgets.IntText(value=0, description="First seed:")
steps = widgets.IntSlider(value=30, min=1, max=50, step=1, description="Steps:")
guidance = widgets.FloatSlider(value=5.0, min=0.0, max=12.0, step=0.5, description="Guidance:")
submit_button = widgets.Button(description="Submit SLURM job", button_style="primary")
refresh_button = widgets.Button(description="Refresh status / show images")
output = widgets.Output()

def update_profile(change=None):
    turbo = model.value == "SDXL Turbo"
    steps.value = 1 if turbo else 30
    guidance.value = 0.0 if turbo else 5.0
    steps.disabled = turbo
    guidance.disabled = turbo
model.observe(update_profile, names="value")
update_profile()

def on_submit(_):
    with output:
        clear_output(wait=True)
        if not prompt.value.strip():
            print("Enter a prompt first.")
            return
        run_name = "batch_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        try:
            submit_job(model.value, prompt.value, count.value, seed.value,
                       steps.value, guidance.value, run_name)
            print("The GPU work is running through SLURM. Click Refresh status / show images later.")
        except Exception as exc:
            print(f"Could not submit job: {type(exc).__name__}: {exc}")

submit_button.on_click(on_submit)
refresh_button.on_click(refresh_jobs)
display(model, prompt, count, seed, steps, guidance, submit_button, refresh_button, output)


### If a job is queued or running

Use the refresh button in the notebook. You can also check the job from a terminal with `squeue -u $USER`; the SLURM log appears under `run_outputs/logs/`. A completed job's images are displayed by the notebook refresh button.